# Leak Checks

This notebook is the central place for HFE system leak analysis. It runs the system $N_2$ pressure-decay test, the main-loop vacuum rate-of-rise test, and the reservoir O-ring leak comparisons through `orca.leaks`.


## Notebook roadmap

1. Locate the repository root and raw data directory.
2. Run the system-pressure leak analysis through `orca.leaks`.
3. Run the main-loop vacuum rate-of-rise analysis with the physically correct fixed-volume model.
4. Display the updated pressure-decay and rate-of-rise figures inline.
5. Run the reservoir O-ring comparison from raw pressure logs.


In [1]:
from pathlib import Path
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import orca
import orca.leaks as leak_tools

importlib.reload(leak_tools)
importlib.reload(orca)

NB_PATH = Path.cwd()
REPO_ROOT = NB_PATH
for candidate in [NB_PATH, *NB_PATH.parents]:
    if (candidate / 'data').exists() and (candidate / 'analysis').exists():
        REPO_ROOT = candidate
        break

TC_CALIBRATION_PATH = REPO_ROOT / 'data' / 'processed' / 'calibration' / 'TC_calibration_20260420.csv'


In [2]:
NB_PATH = Path.cwd()
REPO_ROOT = NB_PATH
for candidate in [NB_PATH, *NB_PATH.parents]:
    if (candidate / 'data').exists() and (candidate / 'analysis').exists():
        REPO_ROOT = candidate
        break

RAW_DIR = REPO_ROOT / 'data' / 'raw'
TC_CALIBRATION_PATH = REPO_ROOT / 'data' / 'processed' / 'calibration' / 'TC_calibration_20260420.csv'

print(f'ORCA: {orca.ORCA_MEANING}')
print(f'Repo root: {REPO_ROOT}')
print(f'TC calibration: {TC_CALIBRATION_PATH}')


ORCA: Operational Recirculation and Cryogenic Analysis
Repo root: /home/aamy/Documents/hfe-system
TC calibration: /home/aamy/Documents/hfe-system/data/processed/calibration/TC_calibration_20260420.csv


## 1. System pressure leak checks

Only include true tank / 1 atm $N_2$ gas-trap runs here. The 2026-03-13 log is included using the tank-pressure channel only (the after-pump sensor was connected to the reservoir that day for the EPDM O-ring test and is ignored here). The 2026-03-13, 2026-04-01, 2026-04-14, 2026-04-20, and 2026-04-23 logs are trimmed to the tank-pressure peak before being overlaid on the system summary.


In [3]:
SYSTEM_POST_FIX_SOURCE_LOG = RAW_DIR / 'leak_test' / 'tc_log_20260319_115916.csv'
SYSTEM_POST_FIX_LOG = REPO_ROOT / 'data' / 'processed' / 'leak_test' / 'log_20260319_post_peak_after_leak_fixes.csv'
APRIL_01_SOURCE_LOG = RAW_DIR / 'leak_test' / 'log_20260401_105911.csv'
APRIL_01_POST_PEAK_LOG = REPO_ROOT / 'data' / 'processed' / 'leak_test' / 'log_20260401_decay.csv'
APRIL_14_SOURCE_LOG = RAW_DIR / 'leak_test' / 'log_20260414_173423.csv'
APRIL_14_POST_PEAK_LOG = REPO_ROOT / 'data' / 'processed' / 'leak_test' / 'log_20260414_decay.csv'
APRIL_20_SOURCE_LOG = RAW_DIR / 'log_20260420_125040.csv'
APRIL_20_POST_PEAK_LOG = REPO_ROOT / 'data' / 'processed' / 'leak_test' / 'log_20260420_decay.csv'
APRIL_23_SOURCE_LOG = RAW_DIR / 'leak_test' / 'log_20260423_172104.csv'
APRIL_23_POST_PEAK_LOG = REPO_ROOT / 'data' / 'processed' / 'leak_test' / 'log_20260423_decay.csv'
MARCH_13_SOURCE_LOG = RAW_DIR / 'leak_test' / 'log_20260313_145244.csv'
MARCH_13_POST_PEAK_LOG = REPO_ROOT / 'data' / 'processed' / 'leak_test' / 'log_20260313_decay.csv'


def backfill_pressure_gauge_columns(frame):
    for abs_column, gauge_column in [
        ('pump_pressure_before_bar_abs', 'pump_pressure_before_bar'),
        ('pump_pressure_after_bar_abs', 'pump_pressure_after_bar'),
        (leak_tools.PRESSURE_ABS_COLUMN, leak_tools.PRESSURE_GAUGE_COLUMN),
    ]:
        if abs_column in frame.columns and gauge_column not in frame.columns:
            frame[gauge_column] = pd.to_numeric(frame[abs_column], errors='coerce') - leak_tools.P_ATM_BAR
    return frame


def export_system_log_slice(frame, output_path):
    sliced_frame = backfill_pressure_gauge_columns(frame.copy())
    sliced_frame[leak_tools.TIME_COLUMN] = pd.to_numeric(
        sliced_frame[leak_tools.TIME_COLUMN],
        errors='coerce',
    )
    sliced_frame = sliced_frame.dropna(subset=[leak_tools.TIME_COLUMN]).copy()
    sliced_frame[leak_tools.TIME_COLUMN] = (
        sliced_frame[leak_tools.TIME_COLUMN] - float(sliced_frame[leak_tools.TIME_COLUMN].iloc[0])
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sliced_frame.to_csv(output_path, index=False)
    return output_path


def build_post_peak_system_log(source_path, output_path, *, pressure_column=leak_tools.PRESSURE_ABS_COLUMN):
    frame = pd.read_csv(source_path, comment='#').dropna(subset=[leak_tools.TIME_COLUMN, pressure_column]).copy()
    peak_index = int(pd.to_numeric(frame[pressure_column], errors='coerce').idxmax())
    return export_system_log_slice(frame.iloc[peak_index:].copy(), output_path)


build_post_peak_system_log(SYSTEM_POST_FIX_SOURCE_LOG, SYSTEM_POST_FIX_LOG)
build_post_peak_system_log(MARCH_13_SOURCE_LOG, MARCH_13_POST_PEAK_LOG)
build_post_peak_system_log(APRIL_01_SOURCE_LOG, APRIL_01_POST_PEAK_LOG)
build_post_peak_system_log(APRIL_14_SOURCE_LOG, APRIL_14_POST_PEAK_LOG)
build_post_peak_system_log(APRIL_20_SOURCE_LOG, APRIL_20_POST_PEAK_LOG)
build_post_peak_system_log(APRIL_23_SOURCE_LOG, APRIL_23_POST_PEAK_LOG)

SYSTEM_PRESSURE_LOGS = [
    RAW_DIR / 'leak_test' / 'log_20260311_114721.csv',
    RAW_DIR / 'leak_test' / 'log_20260312_172440.csv',
    MARCH_13_POST_PEAK_LOG,
    SYSTEM_POST_FIX_LOG,
    APRIL_01_POST_PEAK_LOG,
    APRIL_14_POST_PEAK_LOG,
    APRIL_20_POST_PEAK_LOG,
    APRIL_23_POST_PEAK_LOG,
]

system_results = [leak_tools.analyze_system_pressure_log(path) for path in SYSTEM_PRESSURE_LOGS]
system_result_by_log_file = {result.series.csv_path.name: result for result in system_results}
system_summary = leak_tools.system_pressure_summary_table(system_results)
display(
    system_summary.round(
        {
            'elapsed_h': 3,
            'fit_asymptote_bar_abs': 4,
            'fit_k_per_h': 5,
            'mean_pressure_bar_abs': 3,
            'leak_mbar_l_s': 4,
            'hfe_loss_l_year': 2,
            'rmse_mbar': 1,
        }
    )
)


,log_file,elapsed_h,fit_asymptote_bar_abs,fit_k_per_h,mean_pressure_bar_abs,top_gas_total_pressure_bar_abs,hfe_vapor_partial_pressure_bar_abs,n2_partial_pressure_bar_abs,top_gas_hfe_mole_fraction,leak_mbar_l_s,hfe_loss_l_year,rmse_mbar
0,log_20260311_114721.csv,2.915,1.0132,-0.47146,1.154,1.150758,0.137508,1.01325,0.119494,0.1104,26.24,8.5
1,log_20260312_172440.csv,17.119,1.0132,-0.40822,1.154,1.146849,0.133599,1.01325,0.116492,0.0929,22.10,29.2
2,log_20260313_decay.csv,22.998,1.0132,-0.33882,1.154,1.147172,0.133922,1.01325,0.116741,0.0773,18.39,57.8
3,log_20260319_post_peak_after_leak_fixes.csv,21.180,1.0132,-0.01220,1.154,1.147195,0.133945,1.01325,0.116759,0.0028,0.66,8.4
4,log_20260401_decay.csv,17.119,1.0132,-0.00492,1.154,1.145314,0.132064,1.01325,0.115308,0.0011,0.26,7.8
5,log_20260414_decay.csv,19.806,1.0132,-0.00428,1.154,1.144231,0.130981,1.01325,0.114471,0.0010,0.23,7.8
6,log_20260420_decay.csv,20.635,1.0132,-0.00508,1.154,1.149468,0.136218,1.01325,0.118506,0.0012,0.28,8.7
7,log_20260423_decay.csv,15.518,1.0132,-0.00448,1.154,1.145795,0.132545,1.01325,0.115680,0.0010,0.24,10.0


In [4]:
for result in system_results:
    fig, _ = leak_tools.plot_system_pressure_result(result)
    display(fig)
    plt.close(fig)


<Figure size 1200x800 with 1 Axes>

<Figure size 1200x800 with 1 Axes>

<Figure size 1200x800 with 1 Axes>

<Figure size 1200x800 with 1 Axes>

<Figure size 1200x800 with 1 Axes>

<Figure size 1200x800 with 1 Axes>

<Figure size 1200x800 with 1 Axes>

<Figure size 1200x800 with 1 Axes>

In [5]:
top_system_result = system_result_by_log_file['log_20260312_172440.csv']
march_13_system_result = system_result_by_log_file['log_20260313_decay.csv']
post_fix_system_result = system_result_by_log_file['log_20260319_post_peak_after_leak_fixes.csv']
april_01_summary_result = system_result_by_log_file['log_20260401_decay.csv']
april_14_summary_result = system_result_by_log_file['log_20260414_decay.csv']
april_20_summary_result = system_result_by_log_file['log_20260420_decay.csv']
april_23_summary_result = system_result_by_log_file['log_20260423_decay.csv']


def format_scientific_pm(value, error, *, precision=2):
    if value == 0.0:
        exponent = 0
    else:
        exponent = int(np.floor(np.log10(abs(value))))
    scale = 10.0 ** exponent
    scaled_value = value / scale
    scaled_error = error / scale
    if exponent == 0:
        return rf'({scaled_value:.{precision}f} \pm {scaled_error:.{precision}f})'
    return rf'({scaled_value:.{precision}f} \pm {scaled_error:.{precision}f}) \times 10^{{{exponent}}}'


def system_pressure_drop_metrics(result):
    if result.fit is None:
        return None, None
    fixed_fit, _, _ = leak_tools.fit_fixed_tail_exponential_with_band(
        result.averaged.time_h,
        result.averaged.pressure_abs_bar,
        result.averaged.pressure_sigma_bar,
        result.fit_curve_time_h if result.fit_curve_time_h is not None else result.averaged.time_h,
        asymptote_bar=result.fit.asymptote_bar,
    )
    return leak_tools.compute_start_decay_metrics(fixed_fit)


def system_fit_row(label, result, update_note):
    pressure_drop = '—'
    hfe_loss = '—'
    exp_fit = '—'
    drop_mbar_h, drop_err_mbar_h = system_pressure_drop_metrics(result)
    if drop_mbar_h is not None:
        pressure_drop = f'{drop_mbar_h:.2f} ± {drop_err_mbar_h:.2f}'
    if result.leak is not None:
        hfe_loss = leak_tools.format_leak_annotation_line(result.leak, unit='L/year', use_hfe_loss=True)
    if result.fit is not None:
        initial_pressure_bar = result.fit.asymptote_bar + result.fit.amplitude_bar
        k_text = format_scientific_pm(result.fit.k_per_h, result.fit.k_err_per_h, precision=2)
        exp_fit = (
            rf'P$_{{\infty}}$ = {result.fit.asymptote_bar:.3f} bar, '
            rf'P$_0$ = {initial_pressure_bar:.3f} bar, '
            rf'k = ${k_text}\,\mathrm{{h}}^{{-1}}$'
        )
    return [label, update_note, pressure_drop, hfe_loss, exp_fit]


def plot_system_pressure_summary(traces, *, title, fit_table_rows=None, source_note=None):
    fig = plt.figure(figsize=(12, 10.6))
    grid = fig.add_gridspec(2, 1, height_ratios=[8.0, 2.4], hspace=0.28)
    axis = fig.add_subplot(grid[0])
    table_axis = fig.add_subplot(grid[1])
    table_axis.axis('off')
    y_samples = []
    x_max_h = 0.0

    for trace in traces:
        result = trace['result']
        axis.fill_between(
            result.averaged.time_h,
            result.averaged.pressure_abs_bar - result.averaged.pressure_sigma_bar,
            result.averaged.pressure_abs_bar + result.averaged.pressure_sigma_bar,
            color=trace['color'],
            alpha=trace.get('alpha', 0.18),
            label=trace.get('band_label', '_nolegend_'),
        )
        axis.plot(
            result.averaged.time_h,
            result.averaged.pressure_abs_bar,
            color=trace['color'],
            linewidth=3.0,
            label=trace['line_label'],
        )
        if result.fit_curve_time_h is not None and result.fit_curve_pressure_bar is not None:
            axis.plot(
                result.fit_curve_time_h,
                result.fit_curve_pressure_bar,
                color='black',
                linewidth=2.8,
                linestyle=(0, (6, 3)),
                alpha=0.95,
                zorder=6,
                label=trace.get('fit_label', '_nolegend_'),
            )
            y_samples.append(result.fit_curve_pressure_bar)
        y_samples.extend(
            [
                result.averaged.pressure_abs_bar - result.averaged.pressure_sigma_bar,
                result.averaged.pressure_abs_bar + result.averaged.pressure_sigma_bar,
            ]
        )
        x_max_h = max(x_max_h, float(result.averaged.time_h[-1]))
        if result.fit_curve_time_h is not None:
            x_max_h = max(x_max_h, float(result.fit_curve_time_h[-1]))

    y_all = np.concatenate(y_samples)
    y_min = float(np.min(y_all))
    y_max = float(np.max(y_all))
    y_pad = max(0.04 * (y_max - y_min), 0.03)

    axis.set_xlim(0.0, x_max_h)
    axis.set_ylim(max(leak_tools.P_ATM_BAR, y_min - 0.02), y_max + y_pad)
    axis.set_xlabel('Time (hours)', fontsize=20, labelpad=12)
    axis.set_ylabel('Pressure (bar abs)', fontsize=20)
    axis.set_title(title, fontsize=24)
    axis.tick_params(axis='both', labelsize=16)
    axis.grid(True, alpha=0.3)
    axis.legend(
        loc='lower right',
        bbox_to_anchor=(0.98, 0.24),
        fontsize=16,
        framealpha=0.95,
        ncol=2,
        columnspacing=1.2,
    )

    if fit_table_rows:
        table = table_axis.table(
            cellText=fit_table_rows,
            colLabels=[
                'Dataset',
                'System update done',
                'Pressure drop (mbar/h)',
                'HFE-7200 loss',
                'Exponential Parameters',
            ],
            cellLoc='left',
            colLoc='left',
            bbox=(0.0, 0.02, 1.0, 0.84),
            colWidths=[0.15, 0.15, 0.15, 0.18, 0.37],
        )
        table.auto_set_font_size(False)
        table.set_fontsize(8.8)
        table.scale(1.0, 1.55)
        for (row, col), cell in table.get_celld().items():
            cell.set_edgecolor('0.7')
            cell.PAD = 0.015 if col == 4 else 0.03
            if row == 0:
                cell.set_facecolor('#f2f2f2')
                cell.set_text_props(weight='bold')
            else:
                cell.set_facecolor('white')
                cell.get_text().set_multialignment('left')

    fig.subplots_adjust(left=0.08, right=0.98, top=0.94, bottom=0.10)
    table_position = table_axis.get_position()
    table_axis.set_position([0.02, table_position.y0, 0.96, table_position.height])
    if source_note is not None:
        fig.text(
            0.99,
            0.05,
            source_note,
            ha='right',
            va='bottom',
            fontsize=9,
            color='0.45',
        )
    return fig, axis


system_summary_traces = [
    {
        'result': top_system_result,
        'color': 'tab:blue',
        'line_label': 'Full system (2026-03-12)',
    },
    {
        'result': march_13_system_result,
        'color': 'tab:pink',
        'line_label': 'Full system (2026-03-13)',
    },
    {
        'result': post_fix_system_result,
        'color': 'tab:cyan',
        'line_label': 'Full system (2026-03-19)',
    },
    {
        'result': april_01_summary_result,
        'color': 'tab:olive',
        'line_label': 'Full system (2026-04-01)',
    },
    {
        'result': april_14_summary_result,
        'color': 'tab:brown',
        'line_label': 'Full system (2026-04-14)',
    },
    {
        'result': april_20_summary_result,
        'color': 'tab:gray',
        'line_label': 'Full system (2026-04-20)',
    },
    {
        'result': april_23_summary_result,
        'color': '#005f73',
        'line_label': 'Full system (2026-04-23)',
    },
]

fig, axis = plot_system_pressure_summary(
    system_summary_traces,
    title=r'HFE System $N_2$ Leak Test Summary',
    fit_table_rows=[
        system_fit_row('Full system (2026-03-12)', top_system_result, 'First check'),
        system_fit_row('Full system (2026-03-13)', march_13_system_result, 'FLM bolts retorqued'),
        system_fit_row('Full system (2026-03-19)', post_fix_system_result, 'GTP NPTs redone'),
        system_fit_row('Full system (2026-04-01)', april_01_summary_result, 'RTL rebent'),
        system_fit_row('Full system (2026-04-14)', april_14_summary_result, 'TCs calibration'),
        system_fit_row('Full system (2026-04-20)', april_20_summary_result, 'TCs calibration'),
        system_fit_row('Full system (2026-04-23)', april_23_summary_result, 'Latest log'),
    ],
)

tank_gauge_psi = (top_system_result.averaged.pressure_abs_bar - leak_tools.P_ATM_BAR) * 14.5037738
alignment_order = np.argsort(tank_gauge_psi)
top_system_start_time_h = np.interp(
    24.0,
    tank_gauge_psi[alignment_order],
    top_system_result.averaged.time_h[alignment_order],
)
top_system_time_h = top_system_start_time_h + np.array([0.0, 1.0, 2.0], dtype=float)
top_system_gauge_psi = np.array([24.0, 10.0, 4.1], dtype=float)
top_system_pressure_bar_abs = leak_tools.P_ATM_BAR + top_system_gauge_psi / 14.5037738

top_gauge_min_psi = -30.0 * 0.491154
top_gauge_max_psi = 60.0
top_gauge_span_psi = top_gauge_max_psi - top_gauge_min_psi
top_gauge_quarter_psi = 0.25 * top_gauge_span_psi


def top_gauge_sigma_psi(reading_psig: float) -> float:
    in_end_quarter = (
        reading_psig <= top_gauge_min_psi + top_gauge_quarter_psi
        or reading_psig >= top_gauge_max_psi - top_gauge_quarter_psi
    )
    accuracy_fraction = 0.02 if in_end_quarter else 0.01
    return accuracy_fraction * top_gauge_span_psi / np.sqrt(3.0)


top_system_sigma_bar = np.array(
    [top_gauge_sigma_psi(reading) / 14.5037738 for reading in top_system_gauge_psi],
    dtype=float,
)
bottom_system_start_time_h = top_system_result.averaged.time_h[0]
bottom_system_time_h = bottom_system_start_time_h + np.array([0.0, 1.0, 2.0], dtype=float)
bottom_system_pressure_bar_abs = np.array([2.75, 2.75, 2.75], dtype=float)
bottom_system_sigma_bar = np.full(3, 0.1, dtype=float)
full_system_no_gas_trap_time_h = bottom_system_start_time_h + np.array([0.0, 1.0, 2.0], dtype=float)
full_system_no_gas_trap_pressure_bar_abs = np.array([2.78, 2.78, 2.75], dtype=float)
full_system_no_gas_trap_sigma_bar = np.full(3, 0.1, dtype=float)
gas_trap_gauge_bar = np.array([1.8, 1.78, 1.76, 1.76], dtype=float)
gas_trap_gauge_psi = gas_trap_gauge_bar * 14.5037738
gas_trap_start_time_h = np.interp(
    gas_trap_gauge_psi[0],
    tank_gauge_psi[alignment_order],
    top_system_result.averaged.time_h[alignment_order],
)
gas_trap_time_h = gas_trap_start_time_h + np.array([0.0, 1.0, 2.0, 3.0], dtype=float)
gas_trap_pressure_bar_abs = leak_tools.P_ATM_BAR + gas_trap_gauge_bar
gas_trap_sigma_bar = np.array(
    [top_gauge_sigma_psi(reading) / 14.5037738 for reading in gas_trap_gauge_psi],
    dtype=float,
)

axis.errorbar(
    top_system_time_h,
    top_system_pressure_bar_abs,
    yerr=top_system_sigma_bar,
    fmt='o',
    capsize=4,
    color='tab:orange',
    label='Top of the system',
)
axis.errorbar(
    bottom_system_time_h,
    bottom_system_pressure_bar_abs,
    yerr=bottom_system_sigma_bar,
    fmt='o',
    capsize=4,
    color='tab:red',
    label='Bottom of the system',
)
axis.errorbar(
    full_system_no_gas_trap_time_h,
    full_system_no_gas_trap_pressure_bar_abs,
    yerr=full_system_no_gas_trap_sigma_bar,
    fmt='o',
    capsize=4,
    color='tab:purple',
    label='Full system without gas trap',
)
axis.errorbar(
    gas_trap_time_h,
    gas_trap_pressure_bar_abs,
    yerr=gas_trap_sigma_bar,
    fmt='o',
    capsize=4,
    color='tab:green',
    label='Gas trap after NPT/VCR redo',
)

current_xmin, current_xmax = axis.get_xlim()
current_ymin, current_ymax = axis.get_ylim()
y_measurement_max = max(
    float(np.max(top_system_pressure_bar_abs + top_system_sigma_bar)),
    float(np.max(bottom_system_pressure_bar_abs + bottom_system_sigma_bar)),
    float(np.max(full_system_no_gas_trap_pressure_bar_abs + full_system_no_gas_trap_sigma_bar)),
    float(np.max(gas_trap_pressure_bar_abs + gas_trap_sigma_bar)),
)
axis.set_xlim(current_xmin, max(current_xmax, float(gas_trap_time_h[-1])))
axis.set_ylim(current_ymin, max(current_ymax, y_measurement_max) + 0.03)
axis.legend(
    loc='lower right',
    bbox_to_anchor=(0.99, 0.1),
    fontsize=16,
    framealpha=0.95,
    ncol=2,
    columnspacing=1.2,
)

display(fig)
plt.close(fig)


<Figure size 1200x1060 with 2 Axes>

### Selected HFE System N2 Leak Test Summary

Includes only 2026-03-12, 2026-03-19, and 2026-04-23.

In [6]:
selected_system_summary_traces = [
    trace
    for trace in system_summary_traces
    if trace['line_label'] in {
        'Full system (2026-03-12)',
        'Full system (2026-03-19)',
        'Full system (2026-04-23)',
    }
]

fig, axis = plot_system_pressure_summary(
    selected_system_summary_traces,
    title=r'HFE System $N_2$ Leak Test Summary',
    fit_table_rows=[
        system_fit_row('Full system (2026-03-12)', top_system_result, 'First check'),
        system_fit_row('Full system (2026-03-19)', post_fix_system_result, 'GTP NPTs redone'),
        system_fit_row('Full system (2026-04-23)', april_23_summary_result, 'Latest log'),
    ],
)

axis.set_xlabel('Time (hours)', fontsize=20, labelpad=12)
table_axis = fig.axes[1]
axis_position = axis.get_position()
table_position = table_axis.get_position()
axis_bottom = table_position.y1 + 0.075
axis.set_position([
    axis_position.x0,
    axis_bottom,
    axis_position.width,
    axis_position.y1 - axis_bottom,
])

display(fig)
plt.close(fig)


<Figure size 1200x1060 with 2 Axes>

## 2. Vacuum pressure-rise checks

This is the previous main-loop-only isolated vacuum test. For a fixed isolated volume, the physically correct packaged model is a weighted linear fit in absolute pressure versus time. The water-vapor line is shown as a comparison, not as the primary fit model.


In [7]:
VACUUM_RATE_OF_RISE_CASES = [
    leak_tools.build_vacuum_rate_of_rise_case(
        label='Main Loop Vacuum Test',
        data_inhg=[
            ('2025-10-30 17:30', -28.0),
            ('2025-10-31 17:30', -26.0),
            ('2025-11-03 09:45', -22.0),
            ('2025-11-04 16:30', -20.0),
            ('2025-11-06 11:00', -17.0),
        ],
        volume_l=4.0,
        system_temp_c=21.0,
        source_note='Main-loop vacuum test: 2025-10-30 to 2025-11-06',
    ),
]

vacuum_results = [
    leak_tools.analyze_vacuum_rate_of_rise_case(case)
    for case in VACUUM_RATE_OF_RISE_CASES
]
vacuum_summary = leak_tools.vacuum_rate_of_rise_summary_table(vacuum_results)
display(
    vacuum_summary.round(
        {
            'elapsed_h': 3,
            'start_pressure_bar_abs': 4,
            'end_pressure_bar_abs': 4,
            'rise_rate_mbar_h': 2,
            'gas_load_mbar_l_s': 4,
            'hfe_loss_l_year': 3,
            'water_saturation_bar_abs': 4,
            'start_over_water_sat': 2,
            'end_over_water_sat': 2,
            'rmse_mbar': 1,
        }
    )
)
        


,case,elapsed_h,start_pressure_bar_abs,end_pressure_bar_abs,rise_rate_mbar_h,gas_load_mbar_l_s,hfe_loss_l_year,water_saturation_bar_abs,start_over_water_sat,end_over_water_sat,rmse_mbar
0,Main Loop Vacuum Test,161.5,0.0651,0.4376,2.26,0.0025,0.597,0.0248,2.63,17.66,4.8


In [8]:
for result in vacuum_results:
    fig, _ = leak_tools.plot_vacuum_rate_of_rise_result(result)
    display(fig)
    plt.close(fig)
        


<Figure size 1000x600 with 1 Axes>

In [9]:
VACUUM_PULLDOWN_READING_INHG = -26.0
VACUUM_PULLDOWN_ROOM_TEMP_C = vacuum_results[0].case.system_temp_c
COMPOUND_GAUGE_MIN_PSIG = -30.0 * 0.491154
COMPOUND_GAUGE_MAX_PSIG = 60.0
COMPOUND_GAUGE_SPAN_PSIG = COMPOUND_GAUGE_MAX_PSIG - COMPOUND_GAUGE_MIN_PSIG
COMPOUND_GAUGE_RESOLUTION_INHG = 1.0
INHG_TO_PSI = 0.491154


def compound_grade_a_sigma_inhg(reading_inhg, *, resolution_inhg=COMPOUND_GAUGE_RESOLUTION_INHG):
    reading_psig = reading_inhg * INHG_TO_PSI
    quarter_span_psig = 0.25 * COMPOUND_GAUGE_SPAN_PSIG
    in_end_quarter = (
        reading_psig <= COMPOUND_GAUGE_MIN_PSIG + quarter_span_psig
        or reading_psig >= COMPOUND_GAUGE_MAX_PSIG - quarter_span_psig
    )
    accuracy_fraction = 0.02 if in_end_quarter else 0.01
    accuracy_sigma_psig = accuracy_fraction * COMPOUND_GAUGE_SPAN_PSIG / np.sqrt(3.0)
    resolution_sigma_psig = resolution_inhg * INHG_TO_PSI / np.sqrt(3.0)
    return np.sqrt(accuracy_sigma_psig**2 + resolution_sigma_psig**2) / INHG_TO_PSI


def abs_bar_to_inhg(abs_bar):
    return (abs_bar - leak_tools.P_ATM_BAR) / leak_tools.INHG_TO_BAR


vacuum_pulldown_sigma_inhg = compound_grade_a_sigma_inhg(VACUUM_PULLDOWN_READING_INHG)
vacuum_pulldown_abs_bar = leak_tools.P_ATM_BAR + VACUUM_PULLDOWN_READING_INHG * leak_tools.INHG_TO_BAR
vacuum_pulldown_sigma_bar = vacuum_pulldown_sigma_inhg * leak_tools.INHG_TO_BAR
hfe_only_bar = leak_tools.hfe_vapor_pressure_bar(VACUUM_PULLDOWN_ROOM_TEMP_C)
water_only_bar = leak_tools.water_vapor_pressure_bar(VACUUM_PULLDOWN_ROOM_TEMP_C)
hfe_plus_water_bar = hfe_only_bar + water_only_bar

vacuum_pulldown_comparison = pd.DataFrame(
    [
        {
            'case': 'Measured pulldown',
            'pressure_bar_abs': vacuum_pulldown_abs_bar,
            'equivalent_gauge_inhg': VACUUM_PULLDOWN_READING_INHG,
            'difference_from_measured_mbar': 0.0,
            'difference_sigma': 0.0,
        },
        {
            'case': 'HFE-7200 saturation only',
            'pressure_bar_abs': hfe_only_bar,
            'equivalent_gauge_inhg': abs_bar_to_inhg(hfe_only_bar),
            'difference_from_measured_mbar': 1000.0 * (hfe_only_bar - vacuum_pulldown_abs_bar),
            'difference_sigma': (hfe_only_bar - vacuum_pulldown_abs_bar) / vacuum_pulldown_sigma_bar,
        },
        {
            'case': 'Water saturation only',
            'pressure_bar_abs': water_only_bar,
            'equivalent_gauge_inhg': abs_bar_to_inhg(water_only_bar),
            'difference_from_measured_mbar': 1000.0 * (water_only_bar - vacuum_pulldown_abs_bar),
            'difference_sigma': (water_only_bar - vacuum_pulldown_abs_bar) / vacuum_pulldown_sigma_bar,
        },
        {
            'case': 'HFE-7200 + water saturation',
            'pressure_bar_abs': hfe_plus_water_bar,
            'equivalent_gauge_inhg': abs_bar_to_inhg(hfe_plus_water_bar),
            'difference_from_measured_mbar': 1000.0 * (hfe_plus_water_bar - vacuum_pulldown_abs_bar),
            'difference_sigma': (hfe_plus_water_bar - vacuum_pulldown_abs_bar) / vacuum_pulldown_sigma_bar,
        },
    ]
)
display(
    vacuum_pulldown_comparison.round(
        {
            'pressure_bar_abs': 4,
            'equivalent_gauge_inhg': 2,
            'difference_from_measured_mbar': 1,
            'difference_sigma': 2,
        }
    )
)

vacuum_pulldown_plot_cases = [
    ('Measured pulldown', vacuum_pulldown_abs_bar, 'black'),
    ('HFE-7200 saturation only', hfe_only_bar, 'tab:olive'),
    ('Water saturation only', water_only_bar, 'tab:blue'),
    ('HFE-7200 + water saturation', hfe_plus_water_bar, 'tab:purple'),
]
vacuum_pulldown_plot_y = np.arange(len(vacuum_pulldown_plot_cases), dtype=float)

def inhg_to_abs_bar(reading_inhg):
    return leak_tools.P_ATM_BAR + reading_inhg * leak_tools.INHG_TO_BAR


fig, axis = plt.subplots(figsize=(11.5, 5.8))
axis.axvspan(
    vacuum_pulldown_abs_bar - vacuum_pulldown_sigma_bar,
    vacuum_pulldown_abs_bar + vacuum_pulldown_sigma_bar,
    color='0.88',
    alpha=0.9,
    label='Measured pulldown ±1σ',
)
axis.axvline(
    vacuum_pulldown_abs_bar,
    color='0.35',
    linewidth=2.0,
    linestyle='--',
    label='Measured pulldown mean',
)

for y_value, (label, pressure_bar_abs, color) in zip(vacuum_pulldown_plot_y, vacuum_pulldown_plot_cases):
    marker_size = 130 if label == 'Measured pulldown' else 110
    axis.scatter(
        pressure_bar_abs,
        y_value,
        s=marker_size,
        color=color,
        edgecolor='white',
        linewidth=0.8,
        zorder=4,
    )
    offset_bar = 0.002 if label == 'Measured pulldown' else 0.004
    axis.text(
        pressure_bar_abs + offset_bar,
        y_value,
        f'{pressure_bar_abs:.4f} bar abs\n({abs_bar_to_inhg(pressure_bar_abs):.2f} inHg gauge)',
        va='center',
        ha='left',
        fontsize=10,
        color='0.25',
    )

axis.set_yticks(vacuum_pulldown_plot_y)
axis.set_yticklabels([label for label, _, _ in vacuum_pulldown_plot_cases], fontsize=12)
axis.invert_yaxis()
x_min = min(water_only_bar, vacuum_pulldown_abs_bar - vacuum_pulldown_sigma_bar) - 0.01
x_max = max(hfe_plus_water_bar, vacuum_pulldown_abs_bar + vacuum_pulldown_sigma_bar) + 0.025
axis.set_xlim(x_min, x_max)
axis.set_xlabel('Pressure (bar abs)', fontsize=16)
axis.set_title('Vacuum Pulldown vs Room-Temperature Vapor Pressure Cases', fontsize=20)
axis.grid(True, axis='x', alpha=0.3)
axis.legend(loc='upper left', fontsize=11, framealpha=0.95)
secondary_axis = axis.secondary_xaxis('top', functions=(abs_bar_to_inhg, inhg_to_abs_bar))
secondary_axis.set_xlabel('Equivalent compound-gauge reading (inHg)', fontsize=14)
secondary_axis.tick_params(axis='x', labelsize=11)
fig.subplots_adjust(bottom=0.20)
fig.text(
    0.98,
    0.04,
    (
        'Measured with the -30 inHg to 60 psi Grade A compound gauge\n'
        f'Room-temperature comparison at {VACUUM_PULLDOWN_ROOM_TEMP_C:.1f} C'
    ),
    ha='right',
    va='bottom',
    fontsize=10,
    color='0.4',
)
display(fig)
plt.close(fig)

print(
    'Measured pulldown from the -30 inHg to 60 psi Grade A compound gauge: '
    f'{VACUUM_PULLDOWN_READING_INHG:.1f} ± {vacuum_pulldown_sigma_inhg:.2f} inHg '
    f'(1σ), or {vacuum_pulldown_abs_bar:.4f} ± {vacuum_pulldown_sigma_bar:.4f} bar abs.'
)
print(
    f'Room-temperature assumption for this comparison: {VACUUM_PULLDOWN_ROOM_TEMP_C:.1f} C.'
)
print(
    f'HFE-7200 saturation at that temperature is {hfe_only_bar:.4f} bar abs '
    f'({abs_bar_to_inhg(hfe_only_bar):.2f} inHg gauge).'
)
print(
    f'Water saturation at that temperature is {water_only_bar:.4f} bar abs '
    f'({abs_bar_to_inhg(water_only_bar):.2f} inHg gauge).'
)
print(
    'If both condensed HFE-7200 and water were present, an idealized upper-bound '
    f'saturation pressure is {hfe_plus_water_bar:.4f} bar abs '
    f'({abs_bar_to_inhg(hfe_plus_water_bar):.2f} inHg gauge).'
)
print(
    'Interpretation: the -26 inHg reading is consistent with HFE-7200 saturation within '
    'the compound-gauge uncertainty, is far above the water-only vapor pressure, and does '
    'not cleanly distinguish HFE-only from HFE-plus-water because the gauge uncertainty is large.'
)


,case,pressure_bar_abs,equivalent_gauge_inhg,difference_from_measured_mbar,difference_sigma
0,Measured pulldown,0.1328,-26.00,0.0,0.00
1,HFE-7200 saturation only,0.1372,-25.87,4.4,0.07
2,Water saturation only,0.0248,-29.19,-108.0,-1.72
3,HFE-7200 + water saturation,0.1620,-25.14,29.2,0.47


<Figure size 1150x580 with 1 Axes>

Measured pulldown from the -30 inHg to 60 psi Grade A compound gauge: -26.0 ± 1.85 inHg (1σ), or 0.1328 ± 0.0626 bar abs.
Room-temperature assumption for this comparison: 21.0 C.
HFE-7200 saturation at that temperature is 0.1372 bar abs (-25.87 inHg gauge).
Water saturation at that temperature is 0.0248 bar abs (-29.19 inHg gauge).
If both condensed HFE-7200 and water were present, an idealized upper-bound saturation pressure is 0.1620 bar abs (-25.14 inHg gauge).
Interpretation: the -26 inHg reading is consistent with HFE-7200 saturation within the compound-gauge uncertainty, is far above the water-only vapor pressure, and does not cleanly distinguish HFE-only from HFE-plus-water because the gauge uncertainty is large.


## 3. Reservoir O-ring leak checks

This section keeps both the old sparse reservoir checks and the newer raw logged reservoir runs.

- The legacy sparse-point Viton and FEP results are kept below for reference.
- The raw logged runs use the same averaging and exponential-fit path as the system pressure-drop section.
- If a raw trace does not show a clear pressure drop, the notebook prints a warning and shows only the measured pressure without a fit.


In [10]:
LEGACY_RESERVOIR_VOLUME_L = 11.0
LEGACY_RESERVOIR_OPERATING_GAUGE_BAR = leak_tools.HFE_VAPOR_GAUGE_BAR
PSI_TO_BAR = 0.0689476

legacy_reservoir_cases = [
    leak_tools.ReservoirLeakCase(
        label='FEP O-ring',
        slug='fep_legacy',
        time_h=np.array([0.0, 19.0 + 2.0 / 60.0], dtype=float),
        pressure_abs_bar=np.array(
            [
                23.0 * PSI_TO_BAR + leak_tools.P_ATM_BAR,
                10.0 * PSI_TO_BAR + leak_tools.P_ATM_BAR,
            ],
            dtype=float,
        ),
        volume_l=LEGACY_RESERVOIR_VOLUME_L,
        operating_gauge_bar=LEGACY_RESERVOIR_OPERATING_GAUGE_BAR,
        x_max_h=leak_tools.DEFAULT_RESERVOIR_XMAX_H,
        y_max_bar=leak_tools.DEFAULT_RESERVOIR_YMAX_BAR,
        source_note='Legacy reservoir test data: FEP O-ring (2 spot measurements)',
    ),
    leak_tools.ReservoirLeakCase(
        label='Viton O-ring',
        slug='viton_legacy',
        time_h=np.array([0.0, 4.0 + 47.0 / 60.0], dtype=float),
        pressure_abs_bar=np.array([1.65, 1.35], dtype=float),
        volume_l=LEGACY_RESERVOIR_VOLUME_L,
        operating_gauge_bar=LEGACY_RESERVOIR_OPERATING_GAUGE_BAR,
        x_max_h=leak_tools.DEFAULT_RESERVOIR_XMAX_H,
        y_max_bar=leak_tools.DEFAULT_RESERVOIR_YMAX_BAR,
        source_note='Legacy reservoir test data: Viton O-ring (2 spot measurements)',
    ),
]

legacy_reservoir_results = [
    leak_tools.analyze_reservoir_case(case)
    for case in legacy_reservoir_cases
]
legacy_reservoir_summary = leak_tools.reservoir_summary_table(legacy_reservoir_results)
legacy_reservoir_summary.insert(1, 'data_source', 'legacy sparse points')
display(
    legacy_reservoir_summary.round(
        {
            'elapsed_h': 3,
            'fit_k_per_h': 5,
            'start_pressure_bar_abs': 3,
            'end_pressure_bar_abs': 3,
            'volume_l': 1,
            'operating_pressure_bar_abs': 3,
            'start_decay_mbar_h': 1,
            'leak_mbar_l_s': 4,
            'hfe_loss_l_year': 2,
        }
    )
)


,case,data_source,elapsed_h,fit_k_per_h,start_pressure_bar_abs,end_pressure_bar_abs,volume_l,operating_pressure_bar_abs,start_decay_mbar_h,leak_mbar_l_s,hfe_loss_l_year
0,FEP O-ring,legacy sparse points,19.033,-0.04376,2.599,1.703,11.0,1.176,69.4,0.0218,5.14
1,Viton O-ring,legacy sparse points,4.783,-0.13318,1.650,1.350,11.0,1.176,84.8,0.0663,15.65


In [11]:
for result in legacy_reservoir_results:
    fig, _ = leak_tools.plot_reservoir_leak_result(result)
    display(fig)
    plt.close(fig)


<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

### 3.2 Raw logged reservoir runs

These are the logged reservoir runs used for the Viton, FEP, and EPDM O-ring comparison. The dissertation-facing analysis uses one physical model: exponential relaxation toward atmospheric pressure,

$P_{\mathrm{abs}}(t)=P_{\mathrm{atm}}+A\exp(k t)$,

with $P_{\mathrm{atm}}=1.01325$ bar and a physically valid decay requiring $k<0$. If the pressure decay is not statistically resolved, the result is reported as an HFE-7200 liquid-equivalent upper limit rather than as a separate linear-fit value.

The reservoir measurements use `pump_pressure_after_bar_abs`, because that sensor was connected to the reservoir during these tests. HFE-equivalent losses are computed from the HFE vapor partial-pressure difference at each run's average logged temperature, not from the total reservoir pressure. EPDM 2026-03-16 is retained as supporting raw-log context, while the dissertation comparison and figure use the 2026-03-17 EPDM run.


In [12]:
RESERVOIR_VOLUME_L = 11.0
RESERVOIR_OPERATING_GAUGE_BAR = leak_tools.HFE_VAPOR_GAUGE_BAR
RESERVOIR_PRESSURE_COLUMN = 'pump_pressure_after_bar_abs'
RESERVOIR_PRESSURE_GAUGE_COLUMN = 'pump_pressure_after_bar'
RESERVOIR_BIN_WIDTH_MIN = 1.0

reservoir_log_configs = [
    {
        'label': 'Viton O-ring',
        'slug': 'viton',
        'log_path': RAW_DIR / 'leak_test' / 'tc_log_20260318_110914.csv',
        'pressure_abs_column': RESERVOIR_PRESSURE_COLUMN,
        'pressure_gauge_column': RESERVOIR_PRESSURE_GAUGE_COLUMN,
    },
    {
        'label': 'FEP O-ring',
        'slug': 'fep',
        'log_path': RAW_DIR / 'leak_test' / 'tc_log_20260319_115916.csv',
        'pressure_abs_column': RESERVOIR_PRESSURE_COLUMN,
        'pressure_gauge_column': RESERVOIR_PRESSURE_GAUGE_COLUMN,
    },
    {
        'label': 'EPDM O-ring',
        'slug': 'edpm',
        'log_path': RAW_DIR / 'leak_test' / 'tc_log_20260316_142912.csv',
        'pressure_abs_column': RESERVOIR_PRESSURE_COLUMN,
        'pressure_gauge_column': RESERVOIR_PRESSURE_GAUGE_COLUMN,
    },
]

pending_reservoir_logs = [
    config['label']
    for config in reservoir_log_configs
    if config['log_path'] is None
]
if pending_reservoir_logs:
    print('Pending reservoir raw logs:', ', '.join(pending_reservoir_logs))

reservoir_results = [
    leak_tools.analyze_reservoir_pressure_log(
        csv_path=config['log_path'],
        label=config['label'],
        slug=config['slug'],
        volume_l=RESERVOIR_VOLUME_L,
        operating_gauge_bar=RESERVOIR_OPERATING_GAUGE_BAR,
        pressure_abs_column=config['pressure_abs_column'],
        pressure_gauge_column=config['pressure_gauge_column'],
        bin_width_min=RESERVOIR_BIN_WIDTH_MIN,
    )
    for config in reservoir_log_configs
    if config['log_path'] is not None
]

reservoir_result_by_label = {result.label: result for result in reservoir_results}

if not reservoir_results:
    raise ValueError('Add at least one reservoir raw log path in reservoir_log_configs.')


/home/aamy/Documents/hfe-system/analysis/src/orca/leaks.py:1309: UserWarning: EPDM O-ring: no clear pressure drop; fit skipped.
  ) = analyze_pressure_drop_log(


In [13]:
import warnings


def reservoir_exponential_curve_for_plot(result):
    if result.fit_curve_time_h is not None and result.fit_curve_pressure_bar is not None:
        return result.fit_curve_time_h, result.fit_curve_pressure_bar, result.fit

    curve_time_h = np.linspace(0.0, float(result.averaged.time_h[-1]), 500)
    fixed_tail_fit, curve_pressure_bar, _ = leak_tools.fit_fixed_tail_exponential_with_band(
        result.averaged.time_h,
        result.averaged.pressure_abs_bar,
        result.averaged.pressure_sigma_bar,
        curve_time_h,
        asymptote_bar=leak_tools.P_ATM_BAR,
    )
    return (
        curve_time_h,
        curve_pressure_bar,
        leak_tools.fixed_tail_fit_to_exponential_fit(fixed_tail_fit),
    )


edpm_primary_log = RAW_DIR / 'leak_test' / 'tc_log_20260317_181606.csv'

with warnings.catch_warnings():
    warnings.simplefilter('ignore', UserWarning)
    edpm_primary_result = leak_tools.analyze_reservoir_pressure_log(
        csv_path=edpm_primary_log,
        label='EPDM O-ring',
        slug='edpm_20260317_181606',
        volume_l=RESERVOIR_VOLUME_L,
        operating_gauge_bar=RESERVOIR_OPERATING_GAUGE_BAR,
        pressure_abs_column=RESERVOIR_PRESSURE_COLUMN,
        pressure_gauge_column=RESERVOIR_PRESSURE_GAUGE_COLUMN,
        bin_width_min=RESERVOIR_BIN_WIDTH_MIN,
    )

viton_result = reservoir_result_by_label['Viton O-ring']
fep_result = reservoir_result_by_label['FEP O-ring']

summary_traces = [
    {
        'result': edpm_primary_result,
        'color': 'tab:blue',
        'data_label': 'EPDM data',
    },
    {
        'result': viton_result,
        'color': 'tab:green',
        'data_label': 'Viton data',
    },
    {
        'result': fep_result,
        'color': 'tab:red',
        'data_label': 'FEP data',
    },
]

fig, axis = plt.subplots(figsize=(14.0, 9.5))
x_max_h = min(float(trace['result'].averaged.time_h[-1]) for trace in summary_traces)

for trace in summary_traces:
    result = trace['result']
    lower = result.averaged.pressure_abs_bar - result.averaged.pressure_sigma_bar
    upper = result.averaged.pressure_abs_bar + result.averaged.pressure_sigma_bar
    axis.fill_between(
        result.averaged.time_h,
        lower,
        upper,
        color=trace['color'],
        alpha=0.16,
        linewidth=0.0,
    )
    axis.plot(
        result.averaged.time_h,
        result.averaged.pressure_abs_bar,
        color=trace['color'],
        linewidth=4.0,
        label=trace['data_label'],
    )
    fit_time_h, fit_pressure_bar, _ = reservoir_exponential_curve_for_plot(result)
    axis.plot(
        fit_time_h,
        fit_pressure_bar,
        color='black',
        linewidth=4.0,
        linestyle=(0, (6, 3)),
        label=(
            r'$P_{\mathrm{abs}}(t)=P_{\mathrm{atm}} + A\,\exp(k t)$'
            if trace is summary_traces[0]
            else '_nolegend_'
        ),
    )
axis.set_xlim(0.0, x_max_h)
axis.set_ylim(bottom=leak_tools.P_ATM_BAR, top=2.0)
axis.set_xlabel('Time (hours)', fontsize=32, labelpad=14)
axis.set_ylabel('Reservoir pressure (bar abs)', fontsize=32, labelpad=14)
axis.tick_params(axis='both', labelsize=26)
axis.minorticks_on()
axis.grid(True, which='major', alpha=0.30)
axis.grid(True, which='minor', alpha=0.14, linewidth=0.8)
axis.legend(loc='best', fontsize=22, framealpha=0.95, ncol=1)
fig.tight_layout()
display(fig)
plt.close(fig)


<Figure size 1400x950 with 1 Axes>

In [14]:
viton_result = reservoir_result_by_label['Viton O-ring']

fig, _ = leak_tools.plot_reservoir_pressure_log_result(viton_result)
display(fig)
plt.close(fig)


<Figure size 1200x800 with 1 Axes>

In [15]:
fep_result = reservoir_result_by_label['FEP O-ring']

fig, _ = leak_tools.plot_reservoir_pressure_log_result(fep_result)
display(fig)
plt.close(fig)


<Figure size 1200x800 with 1 Axes>

In [16]:
def format_shared_scientific(value, error, *, precision=2):
    scale_value = max(abs(value), abs(error))
    if scale_value == 0.0:
        exponent = 0
    else:
        exponent = int(np.floor(np.log10(scale_value)))
    scale = 10.0 ** exponent
    return rf'({value / scale:.{precision}f} ± {error / scale:.{precision}f}) × 10^{exponent}'


def format_hfe_loss_result(result):
    if result.leak is not None:
        return (
            f'{result.leak.hfe_loss_l_per_year:.3f} '
            f'± {result.leak.hfe_loss_err_l_per_year:.3f}'
        )
    if result.upper_limit_leak is not None:
        confidence = int(round(100.0 * result.upper_limit_leak.upper_limit_confidence_level))
        return f'<= {result.upper_limit_leak.upper_limit_hfe_loss_l_per_year:.3f} ({confidence}% CL)'
    return 'not resolved'


def format_exponential_k(result):
    if result.fit is None:
        return 'not resolved'
    return f'{format_shared_scientific(result.fit.k_per_h, result.fit.k_err_per_h)} h^-1'


def reservoir_comparison_row(label, result):
    if result.fit is None and result.upper_limit_leak is not None:
        interpretation = 'upper limit only'
    elif result.fit is not None:
        interpretation = 'resolved exponential decay'
    else:
        interpretation = 'not resolved'
    return {
        'Dataset': label,
        'Avg T (C)': f'{result.series.room_temp_c:.3f}',
        'HFE vapor pressure (bar)': f'{leak_tools.hfe_vapor_pressure_bar(result.series.room_temp_c):.6f}',
        'Exponential k (h^-1)': format_exponential_k(result),
        'HFE-7200 loss (L/year)': format_hfe_loss_result(result),
        'Result type': interpretation,
    }


dissertation_reservoir_table = pd.DataFrame(
    [
        reservoir_comparison_row('EPDM (2026-03-17)', edpm_primary_result),
        reservoir_comparison_row('Viton (2026-03-18)', viton_result),
        reservoir_comparison_row('FEP (2026-03-19)', fep_result),
    ]
)
display(dissertation_reservoir_table)


,Dataset,Avg T (C),HFE vapor pressure (bar),Exponential k (h^-1),HFE-7200 loss (L/year),Result type
0,EPDM (2026-03-17),20.736,0.135656,not resolved,<= 0.091 (95% CL),upper limit only
1,Viton (2026-03-18),20.774,0.135882,(-3.65 ± 2.10) × 10^-4 h^-1,0.036 ± 0.021,resolved exponential decay
2,FEP (2026-03-19),20.466,0.134072,(-3.86 ± 0.23) × 10^-3 h^-1,0.376 ± 0.022,resolved exponential decay
